# Study 3, the two-phase P3164 prompt (10 to 19 October 2026): every version, re-scored

Companion to `logs/TWO_PHASE_ABLATION_LOG.md` (the short note). It re-scores the stored runs of every version of the
third prompt study with the lab's exact scorer and reads each arm's pre-registered rule against the numbers.

- **Runs:** `experiments/03_two_phase_csa_pio/lab/runs/two_phase_<version>__v1/gpt-5.6-luna__high/r<k>/` with `phase1/`, `phase2/` and
  `assets/` (one JSON per module), written by `experiments/03_two_phase_csa_pio/two_phase_asset_identification.ipynb` and the `run_*.py` scripts under
  `experiments/03_two_phase_csa_pio/lab/analysis/v10_diagnosis/`. The run folders and the built inputs are local; the public repository does
  not publish them, so a fresh clone scores nothing until they are restored.
- **Reference:** the LAsset authors' manual list (`data/ground_truth/manual_gt_neorv32.json`), primary assets only.
  Findable = reference entries whose name is in the module's closed set: 109 per run on the 15 tuning modules (of 111),
  187 on the 26 held-out modules (of 189), 296 on all 41 (of 300).
- **Scorer:** exact (`compare_v8.score`): a listed name counts when it equals a reference name, each reference entry at
  most once, no credit between a record and its fields. Precision = correct / listed. Recall = correct / findable.
  Pooled = summed over the runs.
- **No API call** unless `RUN_API` is set in the first cell; then the last section runs one version with
  gpt-5.6-luna into its run folder, exactly as the lab notebook does (stored replies are never asked again).
- Three runs per version. The run-to-run spread on the held-out set is about ten correct finds, so a difference
  smaller than that is inside noise.

In [1]:
RUN_API = False      # True: section 8 sends VERSION_TO_RUN to OpenAI. Nothing else calls an API.

import sys, os, io, json, re, subprocess, contextlib
from pathlib import Path
from collections import Counter, defaultdict
import pandas as pd

def _repo_root():
    for q in [Path.cwd(), *Path.cwd().parents]:
        if (q / "FINAL_NOTEBOOK.ipynb").exists():
            return q
    raise SystemExit("start this notebook inside the repository")

ROOT = _repo_root(); os.chdir(ROOT)
LAB = ROOT / "experiments" / "03_two_phase_csa_pio" / "lab"
os.environ["PIO_INPUT_VERSION"] = "v1"            # every run of the study used the built inputs v1
for p in (LAB, LAB / "analysis" / "two_phase_v8__gpt-5.6-luna__high__r0-r2", LAB / "splits",
          ROOT / "assetgen_meta" / "lasset_layer", ROOT / "src", ROOT / "map_pipeline"):
    sys.path.insert(0, str(p))
import labkit as K, chain as CH, check_lab_prompt as CK, cost_estimate as CE, compare_v8 as C8, splits as SP

pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40); pd.set_option("display.max_colwidth", 120)
MODEL, EFF = "gpt-5.6-luna", "high"
G = K.gt_modules()
T15, H26 = list(G["tuning"]), list(G["heldout"])
S2 = SP.load("tuning_v2")                        # tuning_v2 = the 15 + the 12 held-out modules that caused v7's drop
GROUPS = {"tuning 15": T15, "held-out 26": H26, "all 41": T15 + H26, "tuning_v2 27": S2["tuning"], "heldout_v2 14": S2["heldout"]}
REF = C8.refs(sorted(set(T15) | set(H26)))
print("scorer self-test (v7 92/89/86 of 146/158/131; equal to compare_condensed.score on every tuning module and run):",
      "PASS" if C8.selftest(REF) else "FAIL")
print("repo root:", ROOT)
print("findable per run:", {k: sum(len(REF[m]) for m in v) for k, v in GROUPS.items()})

scorer self-test (v7 92/89/86 of 146/158/131; equal to compare_condensed.score on every tuning module and run): PASS
repo root: E:\jobs\ff\test\Prepoison_subset
findable per run: {'tuning 15': 109, 'held-out 26': 187, 'all 41': 296, 'tuning_v2 27': 206, 'heldout_v2 14': 90}


## 1. The chain of versions, and the prompt each run was made with

Each version is a folder `experiments/03_two_phase_csa_pio/lab/prompts/two_phase_<v>/` with `common.md` (shared definitions), `phase1.md`,
`phase2.md` and the two user templates. The system prompt of a phase is `common.md` + `phase<k>.md`. Every stored reply
carries the sha12 of the system prompt it was asked with; the cell recomposes each version's prompts from the folder
and compares. A match means the text in `prompts/` is the text that produced the stored run.

In [2]:
# arm, version, baseline, split the decision was read on, the one change
ARMS = [
    ("A-01", "two_phase_v1",        None,                  "tuning 15",   "the user's definitions; P3164 CSA rubric (Phase 1) and PIO questions (Phase 2)"),
    ("A-02", "two_phase_v2",        "two_phase_v1",        "tuning 15",   "Phase 1 rules A1-A6 (clocks/resets, build-time values, pacing pulses, forwarded transactions, bus handshake copies, working values)"),
    ("A-03", "two_phase_v2a",       "two_phase_v2",        "tuning 15",   "v2 in shorter wording, same meaning"),
    ("A-04", "two_phase_v3",        "two_phase_v2",        "tuning 15",   "specific Phase 1 at fixed length (C1n, C2, C5, C6 hand-offs, C7, 'If the first part holds')"),
    ("A-05", "two_phase_v4",        "two_phase_v3",        "tuning 15",   "C10 in common.md 2.6: bus request/response record ports and their fields are never primary"),
    ("A-06", "two_phase_v4a",       "two_phase_v4",        "tuning 15",   "v4's content in v2a's short wording"),
    ("A-07", "two_phase_v5",        "two_phase_v4a",       "tuning 15",   "rules A (R1 R3 R4 R5 K1 K2), B (R2b R7b), U1-U12 (user decision: remove unlisted extras) + 13 conflict fixes"),
    ("A-08", "two_phase_v5f",       "two_phase_v5",        "tuning 15",   "reply format only: readable keys, each key once, rows grouped by concept, primary/secondary as a label"),
    ("A-09", "two_phase_v5g",       "two_phase_v5f",       "tuning 15",   "Phase 1 / Phase 2 consistency fix in phase2.md only (Phase 1 replies reused)"),
    ("A-10", "two_phase_v6",        "two_phase_v5g",       "tuning 15",   "24 paired Phase 1 / Phase 2 rules from the v5g failure analysis"),
    ("A-11", "two_phase_v6a",       "two_phase_v6",        "tuning 15",   "phase1.md condensed"),
    ("A-12", "two_phase_v6b",       "two_phase_v6",        "tuning 15",   "phase2.md condensed (Phase 1 = v6's replies)"),
    ("A-13", "two_phase_v6c",       "two_phase_v6",        "tuning 15",   "the 35 shared core sentences condensed"),
    ("A-14", "two_phase_v6abc",     "two_phase_v6",        "tuning 15",   "all three condensing tracks; r3-r5 = Phase 2 drawn again on r0-r2's concepts (the spread test)"),
    ("A-15", "two_phase_v6d",       "two_phase_v6",        "tuning 15",   "v6abc + one traced unit restored (Phase 1 = v6abc's replies)"),
    ("A-16", "two_phase_v6e",       "two_phase_v6",        "tuning 15",   "v6 + part of v6b's Phase 2 condensing (Phase 1 = v6's replies)"),
    ("A-17", "two_phase_v7",        "two_phase_v6d",       "tuning 15",   "five narrowings: each rule's exception inside its main sentence (CX2.U3b, CX4.TF1, CX1-ACF, P11.5, LS1)"),
    ("A-18", "two_phase_v8",        "two_phase_v7",        "tuning 15",   "seven rules from the v7 failure analysis (B1, R1-R6)"),
    ("A-19", "two_phase_v9",        "two_phase_v8",        "tuning 15",   "Forwarded transactions rule narrowed (a bus whose address and acknowledge sit on the unit's own non-record ports)"),
    ("A-20", "two_phase_v10",       "two_phase_v9",        "tuning_v2 27", "17 rules rewritten as blocks (Scope, Criteria, Exceptions, Outcome, Evidence) + one shared reading procedure"),
    ("A-21", "two_phase_v11",       "two_phase_v9",        "tuning_v2 27", "merged candidate fixes (Operation selectors last + precedence; bus record-port bar last; Access command fields carried) + N12"),
    ("A-22", "two_phase_v11h",      "two_phase_v9",        "tuning_v2 27", "v11 with only fixes F1 + F2"),
    ("A-23", "two_phase_v12",       "two_phase_v9",        "tuning_v2 27", "the rules that passed alone, merged"),
    ("A-24", "two_phase_v9f",       "two_phase_v9",        "held-out 26", "five Phase 1 rule fixes"),
    ("A-25", "two_phase_v9g",       "two_phase_v9f",       "held-out 26", "+ five keep rules N21-N25 (control record ports, access command ports, entry selects, boundary outputs, coprocessor progress state) + Port buffers revision"),
    ("A-26", "two_phase_v9h",       "two_phase_v9g",       "all 41",      "+ keep blocks K11 Done and wait inputs, K12 Link handshake ports, K13 Busy locks, K14"),
    ("A-27", "two_phase_v9i",       "two_phase_v9h",       "all 41",      "- K14, K12 exception reworded, + five false-positive bars"),
    ("A-28", "two_phase_v9j",       "two_phase_v9i",       "all 41",      "control-unit bar spares the unit's ports and output record; keep blocks moved last"),
    ("A-29", "two_phase_v9k",       "two_phase_v9j",       "tuning_v2 27", "Walk-found internals bar as the last bar of Phase 2 step 4"),
    ("A-30", "two_phase_v9l",       "two_phase_v9j",       "tuning_v2 27", "the walk precedence written into step 4's core sentence"),
    ("A-31", "two_phase_v9m",       "two_phase_v9j",       "tuning_v2 27", "v9l + the code-built LOCATED list as a Phase 2 input"),
    ("A-32", "two_phase_v9n_place", "two_phase_v9j",       "held-out 26", "one primary per place where the information is, in step 4's core sentence (screen A)"),
    ("A-32b", "two_phase_v9n_place2", "two_phase_v9j",     "held-out 26", "screen A2"),
    ("A-32c", "two_phase_v9n_conf", "two_phase_v9j",       "held-out 26", "screen B: confidence key"),
    ("A-32d", "two_phase_v9n_cfg",  "two_phase_v9j",       "held-out 26", "screen C: Phase 1 settings bar"),
    ("A-32e", "two_phase_v9n_dbg",  "two_phase_v9j",       "held-out 26", "screen E: debug bar alone"),
    ("A-33", "two_phase_v9n_place_dbg", "two_phase_v9n_place", "held-out 26", "+ the paired Debug module registers bar (Phase 1 after the control-unit bar, Phase 2 last bar)"),
    ("A-34", "two_phase_p1o_first", "two_phase_v9n_place", "held-out 26", "+ the Debug module registers bar as the FIRST Phase 1 rule block (Phase 2 = v9n_place's file)"),
]
VERSIONS = [a[1] for a in ARMS]
PHASE1_REUSED = {"two_phase_v5g": "two_phase_v5f", "two_phase_v6b": "two_phase_v6", "two_phase_v6d": "two_phase_v6abc", "two_phase_v6e": "two_phase_v6"}

def stored_shas(v):
    # the (phase 1, phase 2) system shas the stored replies carry, over every run of v
    out = {1: set(), 2: set()}
    for r in range(6):
        d = CH.run_dir(v, MODEL, r, EFF)
        if not d.exists():
            continue
        for ph in (1, 2):
            for f in (d / f"phase{ph}").glob("*.json"):
                out[ph].add(json.loads(f.read_text(encoding="utf-8")).get("system sha"))
    return out

rows = []
for arm, v, base, split, what in ARMS:
    s = stored_shas(v)
    here = {ph: CH.sha12(CK.system(v, ph)) for ph in (1, 2)}
    rows.append({"arm": arm, "version": v.replace("two_phase_", ""), "baseline": (base or "").replace("two_phase_", ""), "decided on": split,
                 "phase1 sha": here[1], "phase2 sha": here[2],
                 "stored = recomposed": "no runs" if not (s[1] | s[2]) else ("yes" if s[1] <= {here[1]} and s[2] <= {here[2]} else f"NO {s}"),
                 "words p1/p2/common": "/".join(str(len((LAB / 'prompts' / v / f).read_text(encoding='utf-8').split())) for f in ("phase1.md", "phase2.md", "common.md")),
                 "change": what})
CHAIN = pd.DataFrame(rows)
display(CHAIN)
print("Phase 1 replies reused from another version (so that phase's sha is the other version's):", PHASE1_REUSED)

,arm,version,baseline,decided on,phase1 sha,phase2 sha,stored = recomposed,words p1/p2/common,change
0,A-01,v1,,tuning 15,feac7243a8e8,28a7f02c3a71,yes,1099/2079/4874,the user's definitions; P3164 CSA rubric (Phase 1) and PIO questions (Phase 2)
1,A-02,v2,v1,tuning 15,df44fe84b87d,28a7f02c3a71,yes,1433/2079/4874,"Phase 1 rules A1-A6 (clocks/resets, build-time values, pacing pulses, forwarded transactions, bus handshake copies, ..."
2,A-03,v2a,v2,tuning 15,690b07ad1c3f,50e1406d640d,yes,1288/2079/4057,"v2 in shorter wording, same meaning"
3,A-04,v3,v2,tuning 15,b513d32b7f64,28a7f02c3a71,yes,1506/2079/4874,"specific Phase 1 at fixed length (C1n, C2, C5, C6 hand-offs, C7, 'If the first part holds')"
4,A-05,v4,v3,tuning 15,82e24b9513ae,ca98760b47a5,yes,1506/2079/4926,C10 in common.md 2.6: bus request/response record ports and their fields are never primary
5,A-06,v4a,v4,tuning 15,b13686bbccbf,1c69fe88de91,yes,1448/2079/4109,v4's content in v2a's short wording
6,A-07,v5,v4a,tuning 15,95761173e1f5,0c8ce85ae8d9,yes,2790/2134/4368,"rules A (R1 R3 R4 R5 K1 K2), B (R2b R7b), U1-U12 (user decision: remove unlisted extras) + 13 conflict fixes"
7,A-08,v5f,v5,tuning 15,f05279744fdc,453c149c76b7,yes,2849/2278/4420,"reply format only: readable keys, each key once, rows grouped by concept, primary/secondary as a label"
8,A-09,v5g,v5f,tuning 15,f05279744fdc,57b22998b41f,yes,2849/4606/4420,Phase 1 / Phase 2 consistency fix in phase2.md only (Phase 1 replies reused)
9,A-10,v6,v5g,tuning 15,e815598c3dda,773926fea6fb,yes,6200/7863/4265,24 paired Phase 1 / Phase 2 rules from the v5g failure analysis


Phase 1 replies reused from another version (so that phase's sha is the other version's): {'two_phase_v5g': 'two_phase_v5f', 'two_phase_v6b': 'two_phase_v6', 'two_phase_v6d': 'two_phase_v6abc', 'two_phase_v6e': 'two_phase_v6'}


## 2. What the model was given

Two calls per module. The user message of Phase 1 holds the RTL (comments removed, blank lines left out, each line
numbered as in the source), the closed set (every port and signal of every entity, with record fields, occurrence
lines, role tags and counterparts; sent as compact JSON) and the block diagram that code builds from the RTL's
explicit and implicit flows. Phase 2 gets the same three blocks plus Phase 1's "Conceptual Assets" list as written.
The system prompt grew from v1 to the working version as rules were added; the cell measures both.

In [3]:
M41 = T15 + H26
blk = Counter()
for m in M41:
    d = CH.module_dir(m)
    blk["rtl"] += len((d / "rtl.txt").read_text(encoding="utf-8"))
    blk["closed set (as sent, compact)"] += len(json.dumps(json.loads((d / "closed_set.json").read_text(encoding="utf-8")), separators=(",", ":")))
    blk["diagram"] += len((d / "diagram.txt").read_text(encoding="utf-8"))
tot = sum(blk.values())
print("user-message blocks over the 41 modules (characters):", dict(blk))
print("shares:", {k: f"{v / tot:.1%}" for k, v in blk.items()})
wdt = CH.module_dir("neorv32_wdt")
print("watchdog alone: rtl", len((wdt / "rtl.txt").read_text(encoding="utf-8")), "closed set (compact)",
      len(json.dumps(json.loads((wdt / "closed_set.json").read_text(encoding="utf-8")), separators=(",", ":"))), "diagram", len((wdt / "diagram.txt").read_text(encoding="utf-8")),
      "| Phase 1 user message", len(CH.user_message("two_phase_p1o_first", 1, "neorv32_wdt")))
print()
for v in ("two_phase_v1", "two_phase_v4a", "two_phase_v7", "two_phase_v9", "two_phase_v9j", "two_phase_p1o_first"):
    s1, s2 = CK.system(v, 1), CK.system(v, 2)
    p1 = (LAB / "prompts" / v / "phase1.md").read_text(encoding="utf-8")
    rules = len(re.findall(r"^- [A-Z][^.\n]{2,60}\.", p1, re.M))
    print(f"{v:22s} system prompt chars: phase 1 {len(s1):7,d}  phase 2 {len(s2):7,d}   named rule blocks in phase1.md: {rules}")
print()
print((LAB / "prompts" / "two_phase_p1o_first" / "phase2_user.md").read_text(encoding="utf-8"))

user-message blocks over the 41 modules (characters): {'rtl': 572716, 'closed set (as sent, compact)': 1139390, 'diagram': 139357}
shares: {'rtl': '30.9%', 'closed set (as sent, compact)': '61.5%', 'diagram': '7.5%'}
watchdog alone: rtl 5713 closed set (compact) 14744 diagram 1502 | Phase 1 user message 22203

two_phase_v1           system prompt chars: phase 1  33,961  phase 2  39,552   named rule blocks in phase1.md: 1
two_phase_v4a          system prompt chars: phase 1  32,278  phase 2  35,937   named rule blocks in phase1.md: 7
two_phase_v7           system prompt chars: phase 1  58,561  phase 2  68,286   named rule blocks in phase1.md: 24
two_phase_v9           system prompt chars: phase 1  64,696  phase 2  73,886   named rule blocks in phase1.md: 28
two_phase_v9j          system prompt chars: phase 1  90,206  phase 2  98,361   named rule blocks in phase1.md: 45
two_phase_p1o_first    system prompt chars: phase 1  91,759  phase 2  98,852   named rule blocks in phase1.md: 46

MODUL

## 3. Results: every version, per split

Per complete run (correct, listed), then pooled precision and recall. A version appears on a split only where every
module of that split has an assets file for that run. "all 41" pools the two splits when both have the same runs.
The LAsset rows are the paper's own initial lists (primary assets, one list each), scored with the same scorer on the
same modules: the same pipeline stage, so a target, not a reference.

In [4]:
def score_version(v, mods, reps=range(6)):
    out = []
    for r in reps:
        if not CH.run_dir(v, MODEL, r, EFF).exists():
            continue
        sc = [C8.score(v, r, m, REF) for m in mods]
        if all(s is not None for s in sc):
            out.append({"run": r, "correct": sum(s[1] for s in sc), "listed": sum(s[0] for s in sc),
                        "per module": {m: s[1] for m, s in zip(mods, sc)}, "found": set().union(*(set(s[2]) for s in sc))})
    return out

SC = {v: {g: score_version(v, ms) for g, ms in GROUPS.items()} for v in VERSIONS}
FIND = {g: sum(len(REF[m]) for m in ms) for g, ms in GROUPS.items()}

def pooled(runs, g):
    T, L = sum(x["correct"] for x in runs), sum(x["listed"] for x in runs)
    return (T / L if L else float("nan"), T / (len(runs) * FIND[g]) if runs else float("nan"), T, L)

def cell(v, g):
    runs = SC[v][g]
    if not runs:
        return "-"
    P, R, T, L = pooled(runs, g)
    return f"{P:.3f} / {R:.3f} ({len(runs)}: " + ", ".join(f"{x['correct']}/{x['listed']}" for x in runs) + ")"

rows = [{"arm": a, "version": v.replace("two_phase_", ""), **{g: cell(v, g) for g in GROUPS}} for a, v, *_ in ARMS]
RESULTS = pd.DataFrame(rows)
print("precision / recall (runs: correct/listed per run)")
display(RESULTS)

# LAsset's published initial lists, exact scorer
import lasset_layer as LL
def names_by_module(items):
    out = defaultdict(list)
    if isinstance(items, dict):
        for m, v in items.items():
            out[m] = [x if isinstance(x, str) else next(x[k] for k in ("Asset RTL", "asset_rtl", "rtl", "name", "asset") if k in x) for x in v]
    else:
        for x in items:
            out[next(x[k] for k in ("module", "entity", "ip", "Module") if k in x)].append(next(x[k] for k in ("Asset RTL", "asset_rtl", "rtl", "name", "asset", "structural") if k in x))
    return out
LASSET = {}
for key, label in (("rtl_only", "LAsset RTL-only initial list"), ("spec_rtl", "LAsset spec+RTL initial list")):
    pm = names_by_module(LL.load_list(key))
    row = {"list": label}
    for g, ms in GROUPS.items():
        L = T = 0
        for m in ms:
            km = m if m in pm else m.replace("neorv32_", "")
            listed, c = Counter(pm.get(km, [])), 0
            for n in REF[m]:
                if listed[n] > 0:
                    listed[n] -= 1; c += 1
            L += len(pm.get(km, [])); T += c
        row[g] = f"{T / L:.3f} / {T / FIND[g]:.3f} (1: {T}/{L})"
    LASSET[key] = row
display(pd.DataFrame(LASSET.values()))

precision / recall (runs: correct/listed per run)


,arm,version,tuning 15,held-out 26,all 41,tuning_v2 27,heldout_v2 14
0,A-01,v1,"0.179 / 0.804 (3: 88/480, 87/481, 88/510)","0.239 / 0.822 (3: 164/620, 144/673, 153/633)","0.213 / 0.815 (3: 252/1100, 231/1154, 241/1143)","0.195 / 0.793 (3: 170/831, 158/842, 162/837)","0.264 / 0.867 (3: 82/269, 73/312, 79/306)"
1,A-02,v2,"0.228 / 0.786 (3: 85/425, 85/349, 87/354)","0.244 / 0.779 (3: 153/633, 141/549, 143/609)","0.238 / 0.782 (3: 238/1058, 226/898, 230/963)","0.220 / 0.746 (3: 162/781, 148/626, 151/690)","0.283 / 0.863 (3: 76/277, 78/272, 79/273)"
2,A-03,v2a,"0.210 / 0.798 (3: 85/382, 87/425, 89/434)",-,-,-,-
3,A-04,v3,"0.235 / 0.841 (3: 92/393, 94/371, 89/405)",-,-,-,-
4,A-05,v4,"0.295 / 0.826 (3: 90/286, 91/313, 89/317)",-,-,-,-
5,A-06,v4a,"0.300 / 0.832 (3: 94/320, 84/295, 94/293)","0.275 / 0.788 (3: 154/543, 149/547, 139/517)","0.284 / 0.804 (3: 248/863, 233/842, 233/810)","0.274 / 0.783 (3: 166/611, 155/585, 163/568)","0.306 / 0.852 (3: 82/252, 78/257, 70/242)"
6,A-07,v5,"0.383 / 0.792 (3: 91/246, 81/216, 87/214)","0.318 / 0.740 (3: 140/442, 135/427, 140/438)","0.340 / 0.759 (3: 231/688, 216/643, 227/652)","0.325 / 0.739 (3: 158/487, 144/445, 155/476)","0.377 / 0.804 (3: 73/201, 72/198, 72/176)"
7,A-08,v5f,"0.416 / 0.827 (6: 91/206, 85/214, 92/220, 94/188, 91/244, 88/228)",-,-,-,-
8,A-09,v5g,"0.430 / 0.813 (3: 82/172, 89/222, 95/224)",-,-,-,-
9,A-10,v6,"0.620 / 0.780 (3: 86/143, 83/134, 86/134)","0.426 / 0.652 (3: 120/284, 126/291, 120/285)","0.489 / 0.699 (3: 206/427, 209/425, 206/419)","0.448 / 0.655 (3: 135/309, 133/294, 137/302)","0.590 / 0.800 (3: 71/118, 76/131, 69/117)"


,list,tuning 15,held-out 26,all 41,tuning_v2 27,heldout_v2 14
0,LAsset RTL-only initial list,0.672 / 0.752 (1: 82/122),0.724 / 0.759 (1: 142/196),0.704 / 0.757 (1: 224/318),0.703 / 0.748 (1: 154/219),0.707 / 0.778 (1: 70/99)
1,LAsset spec+RTL initial list,0.730 / 0.917 (1: 100/137),0.730 / 0.866 (1: 162/222),0.730 / 0.885 (1: 262/359),0.716 / 0.869 (1: 179/250),0.761 / 0.922 (1: 83/109)


### 3b. The one input ablation: v4a on inputs v1k (the exact diagram key line)

Inputs v1k differ from v1 only in the ARROWS key line of every diagram (48 lines). The prompt is v4a unchanged, so the
run folder is `two_phase_v4a__v1k`. `labkit` fixes the input version at import, so the cell scores it in a subprocess
with `PIO_INPUT_VERSION=v1k`, with the same scorer, and prints found-in-all-3 for both input versions (the pre-set
reading of that test: this count must not fall).

In [5]:
SCRIPT = '''
import os, sys
from pathlib import Path
from collections import Counter
ROOT = Path(os.environ["PIO_ROOT"]); LAB = ROOT / "experiments/03_two_phase_csa_pio/lab"
for p in (LAB, LAB / "analysis/two_phase_v8__gpt-5.6-luna__high__r0-r2", ROOT / "src"):
    sys.path.insert(0, str(p))
os.chdir(ROOT)
import labkit as K, chain as CH, compare_v8 as C8
T15 = K.gt_modules()["tuning"]; REF = C8.refs(T15)
per = []; found = Counter()
for r in (0, 1, 2):
    sc = [C8.score("two_phase_v4a", r, m, REF) for m in T15]
    assert all(s is not None for s in sc)
    per.append((sum(s[1] for s in sc), sum(s[0] for s in sc)))
    for s in sc:
        for e in s[2]: found[e.split("#")[0]] += 1
T = sum(c for c, l in per); L = sum(l for c, l in per)
print("v4a on inputs", K.INPUT_VERSION, "per run (correct, listed):", per, f"pooled precision {T/L:.3f} recall {T/(3*109):.3f}", "found-in-all-3:", sum(1 for e in found if found[e] == 3))
'''
for iv in ("v1", "v1k"):
    r = subprocess.run([sys.executable, "-I", "-c", SCRIPT], env={**os.environ, "PIO_ROOT": str(ROOT), "PIO_INPUT_VERSION": iv}, capture_output=True, text=True, encoding="utf-8")
    print(r.stdout.strip() or r.stderr[-800:])

v4a on inputs v1 per run (correct, listed): [(94, 320), (84, 295), (94, 293)] pooled precision 0.300 recall 0.832 found-in-all-3: 79
v4a on inputs v1k per run (correct, listed): [(88, 280), (90, 289), (86, 280)] pooled precision 0.311 recall 0.807 found-in-all-3: 68


## 4. Each arm against its baseline, and its pre-registered rule

The rule column is the reading registered before the run (the version README, the run script's docstring, or the
handover log; see the note for sources). Where the rule is numeric it is read off the numbers by code; where it was
read from stored replies by a dedicated script (element lists, rule firing), the verdict is quoted from the log and
marked (log). The module-level sign test counts modules whose summed correct finds rose or fell against the baseline
over the paired runs. Found-in-all-3 counts reference entries found in every run.

In [6]:
import math
def sign_p(a, b):
    n = a + b
    return 1.0 if n == 0 else min(1.0, 2 * sum(math.comb(n, i) for i in range(min(a, b) + 1)) / 2 ** n)

def found_in_all(runs):
    if not runs:
        return None
    c = Counter(e.split("#")[0] for x in runs for e in x["found"])
    return sum(1 for e in c if c[e] == len(runs))

def rr(v, g):      # per-run recall list
    return [x["correct"] / FIND[g] for x in SC[v][g]]
def pp(v, g):      # per-run precision list
    return [x["correct"] / x["listed"] for x in SC[v][g]]

def rule_v2a(v, b, g):
    lo, hi = min(pp(b, g)), max(pp(b, g)); inside = sum(lo <= p <= hi for p in pp(v, g)); return f"precision runs inside v2's range [{lo:.3f}, {hi:.3f}]: {inside} of 3"
def rule_v4a(v, b, g):
    lo = min(rr(b, g)); below = [round(r, 3) for r in rr(v, g) if r < lo]; return f"recall below v4's lowest run {lo:.3f}: {below or 'none'} -> {'fired (r1)' if below else 'held'}"
def rule_v5(v, b, g):
    lo = min(rr(b, g)); below = [round(r, 3) for r in rr(v, g) if r < lo]
    pmin = min(pp(b, g)); pok = all(p > pmin for p in pp(v, g))
    return f"precision above v4a in every run: {pok}; recall below v4a's lowest {lo:.3f}: {below or 'none'} -> {'FAILS the recall test' if below else 'held'}"
def rule_v5f(v, b, g):
    return f"found-in-all-3 {found_in_all(SC[b][g][:3])} -> {found_in_all(SC[v][g][:3])}; P/R up in every run: {all(a > c for a, c in zip(pp(v, g)[:3], pp(b, g)[:3]))} / {all(a > c for a, c in zip(rr(v, g)[:3], rr(b, g)[:3]))}; element rule (3/3 -> 1/3) fired on 4 elements (log)"
def rule_v5g(v, b, g):
    pu = sum(a > c for a, c in zip(pp(v, g), pp(b, g)[3:6])); rn = sum(a >= c for a, c in zip(rr(v, g), rr(b, g)[3:6]))
    return f"against control v5f r3-r5: precision up in {pu} of 3 pairs (need 2), recall not down in {rn} of 3 (need 2)"
def rule_v6(v, b, g):
    return f"precision above 0.500 in {sum(p > 0.5 for p in pp(v, g))} of 3 runs (need 2); recall not below 0.780 in {sum(r >= 0.78 for r in rr(v, g))} of 3 (need 2)"
def rule_cond(v, b, g):
    P, R, T, L = pooled(SC[v][g][:3], g); Pb, Rb, Tb, Lb = pooled(SC[b][g][:3], g)
    return f"pooled vs v6: precision {P:.3f} vs {Pb:.3f}, recall {R:.3f} vs {Rb:.3f} ({T} vs {Tb} correct of 327); Phase 2 spread is 0-5 correct per run"
def rule_v7(v, b, g):
    hi6 = max(rr("two_phase_v6", g)); above = sum(r > hi6 for r in rr(v, g)); below = sum(p < 0.601 for p in pp(v, g))
    return f"recall above v6's highest run {hi6:.3f} in {above} of 3 (2 = supports a gain); precision below 0.601 in {below} run(s) (2 falsifies); the 10 lost elements back in 2+ runs: 6 of 10 (log; cut 5)"
def rule_v8(v, b, g):
    return f"recall below 0.789 in {sum(r < 86/109 for r in rr(v, g))} run(s) (2 falsifies); above 0.844 in {sum(r > 92/109 for r in rr(v, g))} (2 supports); precision below 0.563 in {sum(p < 89/158 for p in pp(v, g))} (2 falsifies); targets found: see compare_v8.py (log)"
def rule_v9(v, b, g):
    return f"tuning {cell(v, g)} vs v8 {cell(b, g)}; R1 xbus correct up on held-out (log: 12 -> 18), R7 falsified (log)"
def rule_v10(v, b, g):
    P, R, *_ = pooled(SC[v][g], g); Pb, Rb, *_ = pooled(SC[b][g], g)
    g2 = "heldout_v2 14"; P2, R2, *_ = pooled(SC[v][g2], g2); P2b, R2b, *_ = pooled(SC[b][g2], g2)
    return f"V1 tuning_v2: precision {P:.3f} vs v9 {Pb:.3f}, recall {R:.3f} vs {Rb:.3f} (cut: neither below v9 by more than 0.02) -> {'PASS' if P >= Pb - 0.02 and R >= Rb - 0.02 else 'FAIL'}; V3 heldout_v2 precision {P2:.3f} vs {P2b:.3f} -> {'PASS' if P2 >= P2b - 0.02 else 'FAIL'}"
def rule_r0_gate(v, b, g):
    x = SC[v][g][0]; base = SC[b][g]
    return f"r0 {x['correct']}/{x['listed']} on {g} vs v9 runs {[(y['correct'], y['listed']) for y in base]}: recall {x['correct'] / FIND[g]:.3f} vs v9 {min(rr(b, g)):.3f}-{max(rr(b, g)):.3f}"
def rule_v9f(v, b, g):
    T = sum(x["correct"] for x in SC[v][g]); Tb = sum(x["correct"] for x in SC[b][g])
    return f"held-out correct over 3 runs {T} vs v9 {Tb} of 561 -> {'does not beat v9' if T <= Tb else 'beats v9'}"
def rule_paired(v, b, g, bar=None):
    runs, base = SC[v][g], SC[b][g]
    n = min(len(runs), len(base)); mods = GROUPS[g]
    up = sum(sum(runs[i]["per module"][m] for i in range(n)) > sum(base[i]["per module"][m] for i in range(n)) for m in mods)
    dn = sum(sum(runs[i]["per module"][m] for i in range(n)) < sum(base[i]["per module"][m] for i in range(n)) for m in mods)
    P, R, T, L = pooled(runs[:n], g); Pb, Rb, Tb, Lb = pooled(base[:n], g)
    s = f"{g}, {n} paired run(s): correct {T} vs {Tb}, precision {P:.3f} vs {Pb:.3f}, recall {R:.3f} vs {Rb:.3f}; modules up {up} down {dn} p {sign_p(up, dn):.2f}"
    return s + (f"; {bar}" if bar else "")
RULES = {
    "two_phase_v1":  ("none (first run): read P, R, Phase 1 kept vs not kept, Phase 2 points per concept", lambda v, b, g: f"found-in-all-3 {found_in_all(SC[v][g])}"),
    "two_phase_v2":  ("precision up with recall held; extra concepts down (log)", rule_paired),
    "two_phase_v2a": ("no structural change: precision inside v2's range (log)", rule_v2a),
    "two_phase_v3":  ("recall up at the same precision; found-in-all-3 up (log)", lambda v, b, g: rule_paired(v, b, g) + f"; found-in-all-3 {found_in_all(SC[b][g])} -> {found_in_all(SC[v][g])}"),
    "two_phase_v4":  ("precision up; bus-record false positives down (log)", rule_paired),
    "two_phase_v4a": ("recall in every run not below v4's range (README falsification rule)", rule_v4a),
    "two_phase_v5":  ("precision up in every run; no run's recall below v4a's range; no element found in every v4a run found in at most one v5 run", rule_v5),
    "two_phase_v5f": ("format slip 0 of 45; element rule: no element 3/3 -> 1/3", rule_v5f),
    "two_phase_v5g": ("vs control v5f r3-r5 on the same Phase 1: precision up in 2 of 3 pairs, recall not down in 2 of 3, no reference element lost", rule_v5g),
    "two_phase_v6":  ("precision above 0.500 in 2 of 3 runs; recall not below 0.780 in 2 or more runs", rule_v6),
    "two_phase_v6a": ("recall and precision level with v6 (falsifiers in README)", rule_cond),
    "two_phase_v6b": ("recall and precision level with v6", rule_cond),
    "two_phase_v6c": ("recall and precision level with v6", rule_cond),
    "two_phase_v6abc": ("recall and precision level with v6", rule_cond),
    "two_phase_v6d": ("comparable to v6 and v6abc -> the final condensed prompt (user)", rule_cond),
    "two_phase_v6e": ("level with v6; tap_reg.dtmcs watch", rule_cond),
    "two_phase_v7":  ("the 10 elements v6 lost recovered in 2+ of 3 runs, at least 5; recall above v6's highest run in 2 runs; precision below 0.601 in at most 1 run", rule_v7),
    "two_phase_v8":  ("compare_v8.py readings 1-5: targets, recall not below 0.789 in 2+ runs, precision not below 0.563 in 2+ runs, B1, element losses", rule_v8),
    "two_phase_v9":  ("compare_v9.py R1-R7: xbus finds back; no loss elsewhere", rule_v9),
    "two_phase_v10": ("V1: tuning_v2 precision and recall each not below v9's by more than 0.02; V3: heldout_v2 precision; V4: watch-list drop at most 2", rule_v10),
    "two_phase_v11": ("r0 gate on tuning_v2 against v9's runs (stage A of PLAN_MERGED.md)", rule_r0_gate),
    "two_phase_v11h": ("r0 gate on tuning_v2: recall not below v9's (PLAN_MERGED.md)", rule_r0_gate),
    "two_phase_v12": ("r0 gate on tuning_v2; targets of the merged rules (N12 fails, log)", rule_r0_gate),
    "two_phase_v9f": ("beats v9 on the held-out 26 over three runs (PLAN_V9F.md)", rule_v9f),
    "two_phase_v9g": ("held-out 26: correct per run above v9f and v9; then the does-not-hurt check on the tuning groups", rule_paired),
    "two_phase_v9h": ("7-module cell then all 41: keep blocks followed, recall up, precision not down", rule_paired),
    "two_phase_v9i": ("precision up on every group, held-out recall not down (r0 only)", rule_paired),
    "two_phase_v9j": ("order B: keep blocks last; all-41 x3 against v9i / v9 (log)", rule_paired),
    "two_phase_v9k": ("gate: tuning_v2 precision above v9j's (walk-found internals barred)", rule_paired),
    "two_phase_v9l": ("gate: tuning_v2 precision above v9j's; held-out recall not down", rule_paired),
    "two_phase_v9m": ("gate: tuning_v2 precision above v9j's; held-out recall not down", rule_paired),
    "two_phase_v9n_place": ("held-out 26: precision up with recall not below v9j's (user recall bar: v9j's 476 of 561)", rule_paired),
    "two_phase_v9n_place2": ("screen: precision up with recall held", rule_paired),
    "two_phase_v9n_conf": ("screen (1 run)", rule_paired),
    "two_phase_v9n_cfg": ("screen (1 run)", rule_paired),
    "two_phase_v9n_dbg": ("screen (2 runs)", rule_paired),
    "two_phase_v9n_place_dbg": ("the debug bar fires (debug module false positives down) with recall held", rule_paired),
    "two_phase_p1o_first": ("Tier 3 cut: recall within noise of A, precision above A; then D107 R3 on the tuning 15: precision above v9j's with recall not below v9j's lowest run", rule_paired),
}
rows = []
for arm, v, base, split, what in ARMS:
    rule, fn = RULES[v]
    try:
        got = fn(v, base, split) if (base and SC[v][split] and SC[base][split]) or not base else ("no runs on this split" if not SC[v][split] else "baseline has no runs here")
    except Exception as e:
        got = f"not readable here: {e}"
    rows.append({"arm": arm, "version": v.replace("two_phase_", ""), "vs": (base or "").replace("two_phase_", ""), "split": split, "pre-registered rule": rule, "got": got})
ARMTABLE = pd.DataFrame(rows)
with pd.option_context("display.max_colwidth", 300):
    display(ARMTABLE)

,arm,version,vs,split,pre-registered rule,got
0,A-01,v1,,tuning 15,"none (first run): read P, R, Phase 1 kept vs not kept, Phase 2 points per concept",found-in-all-3 75
1,A-02,v2,v1,tuning 15,precision up with recall held; extra concepts down (log),"tuning 15, 3 paired run(s): correct 257 vs 263, precision 0.228 vs 0.179, recall 0.786 vs 0.804; modules up 4 down 7 p 0.55"
2,A-03,v2a,v2,tuning 15,no structural change: precision inside v2's range (log),"precision runs inside v2's range [0.200, 0.246]: 3 of 3"
3,A-04,v3,v2,tuning 15,recall up at the same precision; found-in-all-3 up (log),"tuning 15, 3 paired run(s): correct 275 vs 257, precision 0.235 vs 0.228, recall 0.841 vs 0.786; modules up 10 down 2 p 0.04; found-in-all-3 64 -> 77"
4,A-05,v4,v3,tuning 15,precision up; bus-record false positives down (log),"tuning 15, 3 paired run(s): correct 270 vs 275, precision 0.295 vs 0.235, recall 0.826 vs 0.841; modules up 4 down 6 p 0.75"
5,A-06,v4a,v4,tuning 15,recall in every run not below v4's range (README falsification rule),recall below v4's lowest run 0.817: [0.771] -> fired (r1)
6,A-07,v5,v4a,tuning 15,precision up in every run; no run's recall below v4a's range; no element found in every v4a run found in at most one v5 run,precision above v4a in every run: True; recall below v4a's lowest 0.771: [0.743] -> FAILS the recall test
7,A-08,v5f,v5,tuning 15,format slip 0 of 45; element rule: no element 3/3 -> 1/3,found-in-all-3 67 -> 79; P/R up in every run: True / False; element rule (3/3 -> 1/3) fired on 4 elements (log)
8,A-09,v5g,v5f,tuning 15,"vs control v5f r3-r5 on the same Phase 1: precision up in 2 of 3 pairs, recall not down in 2 of 3, no reference element lost","against control v5f r3-r5: precision up in 2 of 3 pairs (need 2), recall not down in 1 of 3 (need 2)"
9,A-10,v6,v5g,tuning 15,precision above 0.500 in 2 of 3 runs; recall not below 0.780 in 2 or more runs,precision above 0.500 in 3 of 3 runs (need 2); recall not below 0.780 in 2 of 3 (need 2)


## 5. The Phase 1 position tests (Phase 1-only runs, read by located share)

From 19 October the position of a Phase 1 block was measured with Phase 1 calls alone (about USD 0.53 per held-out run
against 1.26 for both phases). Located share = reference entries that Phase 1 locates (a port under Enters / Leaves
Through, or an element assigned at a held line; `chain.located`) divided by findable; an upper bound on recall except
for names Phase 2 finds while walking. The readers re-derive every figure from the stored `phase1/` records and start
with their own self-tests (Tier 1: A reproduces 158 / 159 / 152 located).

In [7]:
D = LAB / "analysis" / "v10_diagnosis"
for script in ("read_tier1.py", "read_tier2.py", "read_tier2c.py"):
    print("=" * 30, script, "=" * 30)
    r = subprocess.run([sys.executable, "-B", script], cwd=D, capture_output=True, text=True, encoding="utf-8")
    out = r.stdout
    # the readers print a long JSON block; show the per-arm lines and the paired summary only
    keep = [ln for ln in out.splitlines() if not ln.startswith("   traced") and not ln.startswith("   debug") and ("cites" not in ln)]
    print("\n".join(keep[:60]))
    if r.returncode:
        print("STDERR:", r.stderr[-2000:])

============================== read_tier1.py ==============================


read_tier1 self-test (P0 = A: 158 / 159 / 152 located): PASS
P0 absent (A)              r0 158/187 vol 407 | r1 159/187 vol 405 | r2 152/187 vol 406 || pooled share 0.836 vol 406.0 concepts 170.7 not kept 171.3
P1 first                   r0 154/187 vol 324 | r1 157/187 vol 360 | r2 152/187 vol 325 || pooled share 0.825 vol 336.3 concepts 154.0 not kept 163.0
P2 after control-unit bar  r0 154/187 vol 343 | r1 151/187 vol 365 | r2 148/187 vol 383 || pooled share 0.807 vol 363.7 concepts 157.7 not kept 153.3
P3 last                    r0 151/187 vol 335 | r1 153/187 vol 359 | r2 151/187 vol 334 || pooled share 0.811 vol 342.7 concepts 156.3 not kept 160.7
S sham at P2               r0 157/187 vol 397 | r1 154/187 vol 398 | r2 157/187 vol 413 || pooled share 0.834 vol 402.7 concepts 169.0 not kept 170.3
paired vs P0: {
 "P1 first": {
  "entries located in more runs than A": 14,
  "in fewer": 19,
  "sign test p": 0.487,
  "pooled share": 0.825,
  "A": 0.836,
  "difference": -0.011,
  "gate 

W two_phase_p1o_first r0: located 154/187 = 0.824 vol 324 debug ref 5/6 count 9 | pooled share 0.825
FIX first2             r0: located 154/187 = 0.824 (W r0 0.824; up 5 down 5 p 1.00) vol 349 debug ref 5/6 count 10 reset output 0
O1 bars first          r0: located 159/187 = 0.850 (W r0 0.824; up 10 down 5 p 0.30) vol 356 debug ref 6/6 count 17 reset output 1
O2 keeps first         r0: located 152/187 = 0.813 (W r0 0.824; up 12 down 14 p 0.85) vol 327 debug ref 5/6 count 10 reset output 1
O3 hard-to-easy        r0: located 157/187 = 0.840 (W r0 0.824; up 8 down 5 p 0.58) vol 343 debug ref 6/6 count 11 reset output 1
O4 easy-to-hard        r0: located 160/187 = 0.856 (W r0 0.824; up 11 down 5 p 0.21) vol 368 debug ref 6/6 count 10 reset output 1
O5 rules after OUTPUT  r0: located 161/187 = 0.861 (W r0 0.824; up 12 down 5 p 0.14) vol 391 debug ref 6/6 count 13 reset output 1
============================== read_tier2c.py ==============================


W two_phase_p1o_first: runs [0, 1, 2] located 463/561 = 0.825 volume [324, 360, 325] debug [(5, 9), (5, 9), (5, 11)]
O4 easy-to-hard              runs [0, 1, 2]: located 465/561 = 0.829 (W 0.825); up 14 down 12 p 0.845; volume [368, 335, 343]; debug [(6, 10), (6, 10), (6, 10)]; gate False
O5 rules after OUTPUT        runs [0, 1, 2]: located 467/561 = 0.832 (W 0.825); up 17 down 14 p 0.720; volume [391, 361, 362]; debug [(6, 13), (6, 12), (6, 11)]; gate False
O3 hard-to-easy (r0 only)    runs [0]: located 157/187 = 0.840 (W 0.825); up 17 down 9 p 0.169; volume [343]; debug [(6, 11)]; gate False
O2 keeps first (r0 only)     runs [0]: located 152/187 = 0.813 (W 0.825); up 20 down 19 p 1.000; volume [327]; debug [(5, 10)]; gate False
O1 bars first (r0 only)      runs [0]: located 159/187 = 0.850 (W 0.825); up 20 down 7 p 0.019; volume [356]; debug [(6, 17)]; gate False
FIX first2 (r0 only)         runs [0]: located 154/187 = 0.824 (W 0.825); up 14 down 11 p 0.690; volume [349]; debug [(5, 

## 6. Rule-isolation diagnostics (not arms)

Between v9 and v10 each rule was run alone on each module, Phase 1 and Phase 2 wording, two repeats, to find rules the
model applies inconsistently. These runs produce no precision or recall; their readings (flip shares, Phase 1 / Phase
2 disagreement, over-application) live in `experiments/03_two_phase_csa_pio/lab/analysis/rule_isolation_v9/*.md`. The cell counts what is
stored for each variant so the sizes quoted in the note can be checked.

In [8]:
rows = []
for d in sorted((LAB / "runs").glob("rule_isolation_*")):
    n = sum(1 for _ in d.rglob("*.json"))
    rows.append({"variant": d.name, "stored reply files": n})
display(pd.DataFrame(rows))

,variant,stored reply files
0,rule_isolation_v10e1,164
1,rule_isolation_v10e1__claude1,28
2,rule_isolation_v10f,656
3,rule_isolation_v10f2,328
4,rule_isolation_v10f2__unseen6,24
5,rule_isolation_v10f3,164
6,rule_isolation_v10f3__unseen6,24
7,rule_isolation_v10f4,164
8,rule_isolation_v10f4__unseen6,24
9,rule_isolation_v10f__unseen6,48


## 7. What each version cost

From the usage stored with every reply (uncached input, cached input, output with reasoning) at the gpt-5.6-luna list
prices read on 11 October (`cost_estimate.PRICES`). A version whose Phase 1 replies were copied from another version
paid only its Phase 2 (the Phase 1 column then repeats the other version's spend). Failed calls (`_failed_calls.jsonl`)
are added separately.

In [9]:
rows = []
for arm, v, *_ in ARMS:
    cost = {1: 0.0, 2: 0.0}; calls = 0; failed = 0.0
    for r in range(6):
        d = CH.run_dir(v, MODEL, r, EFF)
        if not d.exists():
            continue
        for ph in (1, 2):
            for f in (d / f"phase{ph}").glob("*.json"):
                u = json.loads(f.read_text(encoding="utf-8")).get("usage") or {}
                if u.get("model", MODEL) == MODEL:
                    cost[ph] += CE.usd(u, MODEL); calls += 1
        for x in CH.failed_calls(d):
            failed += CE.usd(x.get("usage") or {}, MODEL)
    rows.append({"arm": arm, "version": v.replace("two_phase_", ""), "stored calls": calls, "USD phase 1": round(cost[1], 2), "USD phase 2": round(cost[2], 2),
                 "USD stored": round(cost[1] + cost[2], 2), "USD failed calls": round(failed, 2),
                 "phase 1 reused from": PHASE1_REUSED.get(v, "")})
COST = pd.DataFrame(rows)
display(COST)
print("sum over the versions above (stored replies, both phases, Phase 1 copies counted once per source version):",
      round(sum(r["USD stored"] - (r["USD phase 1"] if r["phase 1 reused from"] else 0) for r in rows), 2))

,arm,version,stored calls,USD phase 1,USD phase 2,USD stored,USD failed calls,phase 1 reused from
0,A-01,v1,246,2.20,3.68,5.89,0.00,
1,A-02,v2,246,2.00,3.53,5.53,0.00,
2,A-03,v2a,90,0.80,1.29,2.09,0.00,
3,A-04,v3,90,0.82,1.34,2.16,0.00,
4,A-05,v4,90,0.81,1.41,2.22,0.00,
5,A-06,v4a,246,2.22,3.66,5.88,0.00,
6,A-07,v5,246,2.21,3.49,5.70,0.00,
7,A-08,v5f,180,1.67,2.79,4.46,0.00,
8,A-09,v5g,90,0.85,1.38,2.23,0.00,two_phase_v5f
9,A-10,v6,246,2.18,3.43,5.61,0.00,


sum over the versions above (stored replies, both phases, Phase 1 copies counted once per source version): 144.35


## 8. Running a version (off by default)

With `RUN_API = True`, `API.env` holding `OPENAI_API_KEY=...` in the repository root, the built inputs on disk and a
Python that has the `openai` package (the lab used miniconda's), this cell runs `VERSION_TO_RUN` for the given runs on
the given modules with gpt-5.6-luna at effort high, exactly as the lab notebook's cell 9 did: `chain.run_many` sends
every (run, module) pair at once, stores each reply as it arrives with the prompt's sha, never asks a stored reply
again, retries a rate-limit refusal, and asks a reply that fails the format check once more. A version whose Phase 1
replies are copied from another version needs `analysis/prompt_builds/prepare_phase1_copy.py` first. Three runs of
both phases on the 15 tuning modules are 90 calls, about USD 2 at the working version's size. The stored runs are not
reproduced exactly by rerunning: generation is not deterministic.

In [10]:
VERSION_TO_RUN, RUNS_TO_DO, MODULES_TO_RUN = "two_phase_p1o_first", (0, 1, 2), T15

def preview(jobs, modules):
    total = 0
    for v, rep in jobs:
        out = CH.run_dir(v, MODEL, rep, EFF)
        n = Counter()
        for m in modules:
            f1, f2 = out / "phase1" / f"{m}.json", out / "phase2" / f"{m}.json"
            if f1.exists():
                r1 = json.loads(f1.read_text(encoding="utf-8"))
                want = (CH.sha12(CK.system(v, 1)), CH.sha12(CH.user_message(v, 1, m)))
                if (r1.get("system sha"), r1.get("user sha")) != want:
                    raise RuntimeError(f"{f1} was asked with another message; the run would stop this module")
            n["phase 1 stored" if f1.exists() else "phase 1 to send"] += 1
            n["phase 2 stored" if f2.exists() else "phase 2 to send (at most)"] += 1
        total += n["phase 1 to send"] + n["phase 2 to send (at most)"]
        print(f"  {v} r{rep}: {dict(n)}")
    print(f"  calls to send: {total} (plus at most one format retry per module)")

JOBS = [(VERSION_TO_RUN, r) for r in RUNS_TO_DO]
print("preview (nothing is sent):"); preview(JOBS, MODULES_TO_RUN)
if RUN_API:
    assert CK.check(VERSION_TO_RUN), f"prompt check failed: {VERSION_TO_RUN}"
    res = CH.run_many(JOBS, MODULES_TO_RUN, lambda out: CH.openai(MODEL, EFF, out=out, poll=5), MODEL, EFF, executor="openai", workers=None)
    for v, rep in JOBS:
        print(f"--- {v} r{rep}:", json.dumps(CH.readings(v, MODEL, rep, MODULES_TO_RUN, EFF), indent=1, default=str)[:3000])
    # re-score what was just run
    for g in ("tuning 15", "held-out 26"):
        SC[VERSION_TO_RUN][g] = score_version(VERSION_TO_RUN, GROUPS[g])
        print(g, cell(VERSION_TO_RUN, g))
else:
    print("RUN_API is False: nothing sent")

preview (nothing is sent):
  two_phase_p1o_first r0: {'phase 1 stored': 15, 'phase 2 stored': 15}
  two_phase_p1o_first r1: {'phase 1 stored': 15, 'phase 2 stored': 15}
  two_phase_p1o_first r2: {'phase 1 stored': 15, 'phase 2 stored': 15}
  calls to send: 0 (plus at most one format retry per module)
RUN_API is False: nothing sent
